# 17 · `gl_engine/rating/referrals.py`

## What this file is for

**When the engine must not answer.**

ISO's content contains values that are not values — a `0` that means *refer to the company*, a factor ISO leaves to the carrier, a table that is empty because the coverage isn't offered. Producing a number in those situations is worse than producing nothing, because the number looks fine.

This file is the register of every such case, wired to detectors that catch them at rating time.

**Depends on:** [`03-domain-cell`](03-domain-cell.ipynb), [`16-rating-kernel`](16-rating-kernel.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.rating import referrals

for name, obj in vars(referrals).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != referrals.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

The register itself.

In [ ]:
from gl_engine.rating import referrals as R

register = R.load_register()
print(f"{len(register)} register entries\n")

for entry in register[:5]:
    print(f"  {entry['id']:<5} {entry['kind']:<14} {str(entry['condition'])[:70]}")

## The interesting case

### Every entry has an explicit disposition

Nothing is left undecided by accident. An entry is either enforced, deliberately not a referral, supplied as configuration, or still pending — and pending is visible rather than silent.

In [ ]:
from collections import Counter
from gl_engine.rating import Kernel

counts = Counter(R.disposition(e["id"]) for e in register)
for k, v in sorted(counts.items()):
    print(f"  {k:<14} {v}")

kernel = Kernel()
print(f"\nenforced at rating time : {len(kernel.enforced)}")
print(f"still pending           : {len(kernel.unenforced)}")

`CONFIG` is worth understanding: **R18**, the loss cost multiplier, isn't a rating-time referral at all. It's a carrier parameter supplied once at configuration. The engine refuses to rate when it was never supplied — and never refers merely because it resolved to `1`.

### The detectors

Each one catches a specific shape of wrongness, and the names say what they look for.

In [ ]:
import inspect

detectors = [o for n, o in vars(R).items()
             if n.startswith("d_") and inspect.isfunction(o)]
for d in detectors:
    first = (d.__doc__ or "").strip().splitlines()[0] if d.__doc__ else ""
    print(f"  {d.__name__:<32} {first[:72]}")

### On a real rating

In [ ]:
rating = Kernel().rate("../Engine_Payloads/GA/submission.json")
findings = R.run_detectors(rating)

print(f"detectors run over a Georgia rating: {len(findings)} finding(s)")
for f in findings[:5]:
    print("  ", f)
if not findings:
    print("  (clean -- this risk shape triggers none of them)")

## What it refuses

A referral is raised as an exception only when something tries to *use* the value — the same rule as notebook 03.

In [ ]:
from gl_engine.errors import ReferToCompany

print(ReferToCompany.__doc__)
print()
print("A REFER travels intact until the moment it would have become a premium.")
print("That is why a referral is data here and an exception there.")

## Try it yourself

1. Which register entries are still `PENDING`? What would it take to enforce one?
2. `d_negative_factor` exists because a negative premium escaped once. Find the conditions under which it fires.
3. Rate in `underwriting` mode and compare findings against `strict-erc`.

In [ ]:
# your turn